# Layer 3 — NLI Verification (Jathin)
**Pipeline Role:** Uses a pretrained HuggingFace NLI model to check if the article's claim **contradicts known facts**.

**Output format (agreed contract):**
```python
result = {
    'label': 'FAKE',       # or 'REAL'
    'confidence': 0.87,    # float between 0 and 1
    'reason': 'short explanation'
}
```

## 1. Install Dependencies

In [ ]:
!pip install transformers torch sentencepiece --quiet

## 2. Imports

In [ ]:
from transformers import pipeline
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful")

## 3. Load NLI Model

Using `facebook/bart-large-mnli` — a strong zero-shot NLI classifier that works without any fine-tuning on news data.

In [ ]:
print("Loading NLI model (facebook/bart-large-mnli)...")

nli_pipeline = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1  # use CPU; set to 0 if GPU is available
)

print("✅ NLI model loaded successfully")

## 4. Known Fact Reference Statements

These are short, verifiable factual claims used as **hypotheses** in NLI.
The model checks whether the article *entails*, *contradicts*, or is *neutral* to each fact.

> 💡 You can expand this list — more facts = more robust verification.

In [ ]:
KNOWN_FACTS = [
    # Science & Health
    "Vaccines are safe and effective at preventing diseases.",
    "Climate change is caused by human activity and is scientifically proven.",
    "The Earth is approximately 4.5 billion years old.",
    "COVID-19 is caused by the SARS-CoV-2 coronavirus.",
    "Antibiotics do not work against viral infections.",

    # General world facts
    "The United States has 50 states.",
    "World War II ended in 1945.",
    "The Moon orbits the Earth.",
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "The human body has 206 bones.",

    # Media / Misinformation signals
    "News articles that use excessive capital letters or emotional language are often unreliable.",
    "Credible news sources cite named experts and provide verifiable references.",
]

print(f"✅ {len(KNOWN_FACTS)} reference facts loaded")

## 5. NLI Scoring Logic

**How it works:**
- For each known fact, we run NLI with the article as the **premise** and the fact as the **hypothesis**.
- The model returns scores for: `entailment`, `neutral`, `contradiction`.
- A high average **contradiction** score → article likely **FAKE**.
- A high average **entailment** score → article likely **REAL**.

In [ ]:
def truncate_text(text, max_words=200):
    """Truncate article to avoid token limit issues with NLI model."""
    words = text.split()
    if len(words) > max_words:
        return " ".join(words[:max_words]) + "..."
    return text


def compute_nli_scores(article_text, facts, nli_model):
    """
    Run NLI between article (premise) and each known fact (hypothesis).
    Returns per-fact scores and aggregated contradiction/entailment averages.
    """
    premise = truncate_text(article_text)

    contradiction_scores = []
    entailment_scores   = []
    detail_log          = []

    for fact in facts:
        output = nli_model(
            sequences=premise,
            candidate_labels=["entailment", "neutral", "contradiction"],
            hypothesis_template="{}"  # fact is used directly as label text
        )

        score_map = dict(zip(output["labels"], output["scores"]))
        contradiction_scores.append(score_map.get("contradiction", 0))
        entailment_scores.append(score_map.get("entailment", 0))
        detail_log.append({
            "fact"           : fact,
            "contradiction"  : round(score_map.get("contradiction", 0), 4),
            "entailment"     : round(score_map.get("entailment", 0), 4),
            "neutral"        : round(score_map.get("neutral", 0), 4),
        })

    return {
        "mean_contradiction" : float(np.mean(contradiction_scores)),
        "mean_entailment"    : float(np.mean(entailment_scores)),
        "max_contradiction"  : float(np.max(contradiction_scores)),
        "details"            : detail_log,
    }


print("✅ Scoring functions defined")

## 6. Main Layer Function

This is the **single entry point** that the ensemble layer will call.

In [ ]:
def layer3_predict(article_text: str) -> dict:
    """
    Layer 3 — NLI Verification
    
    Parameters
    ----------
    article_text : str
        Raw news article string.

    Returns
    -------
    dict with keys: 'label', 'confidence', 'reason'
    """

    if not article_text or not article_text.strip():
        return {
            "label"     : "REAL",
            "confidence": 0.5,
            "reason"    : "Empty input — defaulting to REAL with low confidence"
        }

    # ── Run NLI against all known facts ──────────────────────────────────────
    scores = compute_nli_scores(article_text, KNOWN_FACTS, nli_pipeline)

    mean_contradiction = scores["mean_contradiction"]
    mean_entailment    = scores["mean_entailment"]
    max_contradiction  = scores["max_contradiction"]

    # ── Decision rule ────────────────────────────────────────────────────────
    # Primary signal: mean contradiction vs mean entailment
    # Tie-break:      max_contradiction catches articles that strongly
    #                 contradict even a single important fact

    FAKE_THRESHOLD = 0.35   # tune this on your validation set

    if mean_contradiction > mean_entailment or max_contradiction > 0.65:
        label = "FAKE"
        # Confidence = how much more contradiction dominates over entailment
        raw_conf = mean_contradiction + 0.5 * (mean_contradiction - mean_entailment)
    else:
        label = "REAL"
        raw_conf = mean_entailment + 0.5 * (mean_entailment - mean_contradiction)

    # Clip confidence to [0.50, 0.99] — never be overconfident or under 50 %
    confidence = float(np.clip(raw_conf, 0.50, 0.99))

    # ── Build reason string ──────────────────────────────────────────────────
    # Find the fact with the highest contradiction score for explainability
    top_fact = max(scores["details"], key=lambda x: x["contradiction"])

    if label == "FAKE":
        reason = (
            f"NLI detected contradiction with known facts "
            f"(mean_contradiction={mean_contradiction:.2f}, "
            f"max={max_contradiction:.2f}). "
            f"Strongest conflict: '{top_fact['fact'][:60]}...'"
        )
    else:
        reason = (
            f"NLI found article consistent with known facts "
            f"(mean_entailment={mean_entailment:.2f}, "
            f"mean_contradiction={mean_contradiction:.2f})"
        )

    return {
        "label"     : label,
        "confidence": round(confidence, 4),
        "reason"    : reason
    }


print("✅ layer3_predict() is ready")

## 7. Quick Smoke Test

In [ ]:
# ── Test 1: Clearly fake article ─────────────────────────────────────────────
fake_article = """
SHOCKING: Scientists have PROVEN that vaccines cause autism and the government 
has been covering this up for decades. A secret study shows that 5G towers are 
responsible for spreading COVID-19, and climate change is a hoax invented by 
globalists to control the population. The Earth is only 6,000 years old according 
to newly discovered documents suppressed by NASA.
"""

result_fake = layer3_predict(fake_article)
print("=== FAKE ARTICLE TEST ===")
print(f"  Label      : {result_fake['label']}")
print(f"  Confidence : {result_fake['confidence']}")
print(f"  Reason     : {result_fake['reason']}")
print()

In [ ]:
# ── Test 2: Clearly real article ─────────────────────────────────────────────
real_article = """
Health authorities confirmed that the latest round of flu vaccinations has proven 
effective in reducing hospitalisation rates among vulnerable populations. Researchers 
at multiple universities corroborated findings that following standard hygiene 
practices significantly slows the spread of respiratory viruses. Climate scientists 
also released a report showing that global temperatures continue to rise due to 
increased greenhouse gas emissions from industrial activity.
"""

result_real = layer3_predict(real_article)
print("=== REAL ARTICLE TEST ===")
print(f"  Label      : {result_real['label']}")
print(f"  Confidence : {result_real['confidence']}")
print(f"  Reason     : {result_real['reason']}")

## 8. Batch Evaluation (Optional — for validation dataset)

In [ ]:
def batch_evaluate(articles: list[dict]) -> list[dict]:
    """
    Run layer3_predict on a list of {text, true_label} dicts.
    Returns predictions + basic accuracy stats.
    
    Each item in articles should be:
        {'text': '...', 'true_label': 'FAKE' or 'REAL'}
    """
    results = []
    correct = 0

    for i, item in enumerate(articles):
        pred = layer3_predict(item["text"])
        pred["true_label"] = item["true_label"]
        pred["correct"]    = pred["label"] == item["true_label"]
        correct += pred["correct"]
        results.append(pred)
        print(f"[{i+1}/{len(articles)}] True: {item['true_label']} | Pred: {pred['label']} | ✅" if pred["correct"] else
              f"[{i+1}/{len(articles)}] True: {item['true_label']} | Pred: {pred['label']} | ❌")

    accuracy = correct / len(articles)
    print(f"\n📊 Accuracy: {accuracy:.2%} ({correct}/{len(articles)})")
    return results


# ── Plug in your own dataset here ────────────────────────────────────────────
# sample_data = [
#     {"text": "...", "true_label": "FAKE"},
#     {"text": "...", "true_label": "REAL"},
# ]
# batch_evaluate(sample_data)

print("✅ batch_evaluate() defined — plug in your dataset to run evaluation")

## 9. Ensemble-Ready Export

When merging with Layer 1 and Layer 2, the ensemble notebook should simply call:
```python
from layer3_jathin import layer3_predict

result = layer3_predict(article_text)
# result = {'label': 'FAKE', 'confidence': 0.87, 'reason': '...'}
```

Or if keeping everything inside notebooks, copy-paste the `layer3_predict` cell into the ensemble notebook.

In [ ]:
# ── Final contract verification ───────────────────────────────────────────────
test_input = "some news article string"
output = layer3_predict(test_input)

assert isinstance(output["label"], str) and output["label"] in {"FAKE", "REAL"}
assert isinstance(output["confidence"], float) and 0 <= output["confidence"] <= 1
assert isinstance(output["reason"], str)

print("✅ Output contract verified — Layer 3 is ensemble-ready")
print(f"   Sample output: {output}")

---
## Summary

| Component | Detail |
|---|---|
| **Model** | `facebook/bart-large-mnli` (zero-shot NLI) |
| **Approach** | Compare article against 12 known facts using NLI entailment vs contradiction |
| **FAKE signal** | High mean contradiction OR max contradiction > 0.65 |
| **REAL signal** | High mean entailment, low contradiction |
| **Output** | `{'label', 'confidence', 'reason'}` — matches agreed contract |
| **Threshold** | `FAKE_THRESHOLD = 0.35` — tune on validation set |
| **Ensemble hook** | Call `layer3_predict(article_text)` from ensemble notebook |